In [0]:
from pyspark.sql.functions import col, trim, lower, datediff, year, month, when

In [0]:
df = spark.table("ecommerce_dev.silver.orders_clean")

orders_enhanced = df \
    .filter(col("order_purchase_timestamp").isNotNull()) \
    .filter(col("order_delivered_customer_date").isNull() | 
            (col("order_delivered_customer_date") >= col("order_purchase_timestamp"))) \
    .withColumn(
        "delivery_days",
        datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))
    ) \
    .withColumn("order_year", year(col("order_purchase_timestamp"))) \
    .withColumn("order_month", month(col("order_purchase_timestamp"))) \
    .withColumn("is_delivered", col("order_status") == "delivered")

orders_enhanced.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_dev.silver.orders_enhanced")

In [0]:
df = spark.table("ecommerce_dev.silver.customers_clean")

customers_enhanced = df \
    .withColumn("customer_city", trim(lower(col("customer_city")))) \
    .withColumn("customer_state", trim(lower(col("customer_state")))) \
    .filter(col("customer_zip_code_prefix").isNotNull())

customers_enhanced.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_dev.silver.customers_enhanced")

In [0]:
df = spark.table("ecommerce_dev.silver.products_clean")

products_enhanced = df \
    .fillna({
        "product_weight_g": 0,
        "product_length_cm": 0,
        "product_height_cm": 0,
        "product_width_cm": 0
    }) \
    .withColumn("product_category_name", trim(lower(col("product_category_name"))))

products_enhanced.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_dev.silver.products_enhanced")

In [0]:
df = spark.table("ecommerce_dev.silver.sellers_clean")

sellers_enhanced = df \
    .withColumn("seller_city", trim(lower(col("seller_city")))) \
    .withColumn("seller_state", trim(lower(col("seller_state"))))

sellers_enhanced.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_dev.silver.sellers_enhanced")

In [0]:
df = spark.table("ecommerce_dev.silver.payments_clean")

payments_enhanced = df \
    .filter(col("payment_value") > 0) \
    .withColumn("payment_type", trim(lower(col("payment_type")))) \
    .withColumn("is_installment", col("payment_installments") > 1)

payments_enhanced.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_dev.silver.payments_enhanced")

In [0]:
df = spark.table("ecommerce_dev.silver.reviews_clean")

reviews_enhanced = df \
    .withColumn(
        "review_category",
        when(col("review_score") >= 4, "positive")
        .when(col("review_score") == 3, "neutral")
        .otherwise("negative")
    )

reviews_enhanced.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_dev.silver.reviews_enhanced")

In [0]:
df = spark.table("ecommerce_dev.silver.order_items_clean")

items_enhanced = df \
    .filter(col("price") > 0) \
    .filter(col("freight_value") >= 0) \
    .withColumn("total_item_value", col("price") + col("freight_value"))

items_enhanced.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_dev.silver.order_items_enhanced")

In [0]:
df = spark.table("ecommerce_dev.silver.geolocation_clean")

geo_enhanced = df \
    .withColumn("geolocation_city", trim(lower(col("geolocation_city")))) \
    .withColumn("geolocation_state", trim(lower(col("geolocation_state")))) \
    .withColumn("geolocation_lat", col("geolocation_lat").cast("double")) \
    .withColumn("geolocation_lng", col("geolocation_lng").cast("double"))

geo_enhanced.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_dev.silver.geolocation_enhanced")

In [0]:
df = spark.table("ecommerce_dev.silver.category_translation_clean")

category_enhanced = df \
    .withColumn("product_category_name", trim(lower(col("product_category_name")))) \
    .withColumn("product_category_name_english", trim(lower(col("product_category_name_english"))))

category_enhanced.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_dev.silver.category_translation_enhanced")